# 22.3 用 FastAPI 部署模型:把模型变成服务 / Deploying Models with FastAPI

**中文**:模型存下来了(22.1)、打包成 Pipeline 了(22.2),但它还只是磁盘上的一个文件。要让**别的系统、App、前端**能用它,得把它变成一个**在线服务**——通过网络接收数据、返回预测。事实标准是把模型包成一个 **REST API**:客户端发一个 HTTP 请求(带特征),服务返回一个 JSON(带预测)。**FastAPI** 是当今 Python 做这件事的首选(高性能、异步、自动数据校验、自动生成 API 文档)。但比"会写 FastAPI 语法"更重要的,是理解一个模型服务**到底要做对哪几件事**:请求校验、模型只加载一次、健康检查、错误处理、延迟。本节从零实现一个 mini 模型服务(把 FastAPI+pydantic 的核心逻辑用纯 Python 复现),再给出真实的 FastAPI 参考代码——让你既懂原理又能上手。
**English**: The model is saved (22.1) and packaged as a Pipeline (22.2), but it's still just a file on disk. For **other systems, apps, and frontends** to use it, you must turn it into an **online service** — receiving data and returning predictions over the network. The de-facto standard is wrapping the model in a **REST API**: a client sends an HTTP request (with features), the service returns JSON (with a prediction). **FastAPI** is today's top Python choice for this (high-performance, async, automatic validation, auto-generated API docs). But more important than "knowing FastAPI syntax" is understanding **what a model service must get right**: request validation, loading the model once, health checks, error handling, latency. This section builds a mini model service from scratch (reproducing FastAPI + pydantic's core logic in pure Python), then gives real FastAPI reference code — so you understand the principles and can get hands-on.

---

**中文**:**一个模型服务的请求生命周期**(每个环节都可能出错):
**English**: **The request lifecycle of a model service** (each step can fail):
1. **中文**:**接收请求** → 客户端 POST 一个 JSON(如 `{"sepal_length": 5.1, ...}`)。
   **Receive request** → client POSTs a JSON (e.g. `{"sepal_length": 5.1, ...}`).
2. **中文**:**校验输入(关键!)** → 检查字段齐全、类型正确、取值合理。**永远不要信任外部输入**——缺字段、类型错、超范围都要**明确拒绝(返回 422)**,而不是让模型崩溃或给出垃圾预测。这是 pydantic 的职责。
   **Validate input (critical!)** → check fields present, types correct, values sensible. **Never trust external input** — missing fields, wrong types, out-of-range must be **explicitly rejected (return 422)**, not crash the model or produce garbage. This is pydantic's job.
3. **中文**:**预测** → 用**启动时就加载好的**模型(绝不每次请求都从磁盘加载!)算出结果。
   **Predict** → use the model **loaded at startup** (never load from disk per request!) to compute the result.
4. **中文**:**返回响应** → 结构化 JSON(预测值 + 置信度 + 元数据),合适的 HTTP 状态码。
   **Return response** → structured JSON (prediction + confidence + metadata), appropriate HTTP status code.

**中文**:**为什么用 FastAPI**:①**自动校验**——你用 Python 类型注解(pydantic 模型)声明输入长啥样,FastAPI 自动校验、自动返回清晰的错误;②**异步高性能**——`async` 支持高并发,配 uvicorn/gunicorn 多 worker;③**自动文档**——自动生成交互式 API 文档(Swagger UI),前端/调用方直接看着调;④**类型安全**——请求/响应都是强类型,少踩运行时坑。
**English**: **Why FastAPI**: ① **automatic validation** — you declare the input shape with Python type hints (pydantic models), and FastAPI auto-validates and returns clear errors; ② **async high performance** — `async` supports high concurrency, with uvicorn/gunicorn multi-worker; ③ **auto docs** — auto-generated interactive API docs (Swagger UI) for frontend/callers; ④ **type safety** — requests/responses are strongly typed, fewer runtime traps.

> 💡 **面试速查 / Interview cheat-sheet（★★ 部署必考）**
> **中文**:**模型服务=把模型包成 REST API**(客户端 POST 特征 JSON→返回预测 JSON), 解耦模型与调用方、语言无关、可独立扩缩。**FastAPI** 优势:pydantic 自动校验输入、async 高并发、自动 Swagger 文档、类型安全。**必做对的几件事**:①**模型启动时加载一次**(全局/lifespan, 绝不每请求从磁盘读→否则延迟爆炸)②**严格校验输入**(缺字段/类型错/越界→返回 422, 别信任外部输入)③**健康检查端点**(`/health` 给 K8s 探活, readiness vs liveness)④**结构化响应+状态码**(200/422/500)⑤**低延迟**(批量推理、异步、模型量化)。**部署栈**:FastAPI + **uvicorn/gunicorn**(ASGI 服务器, 多 worker)+ 容器(22.5)+ K8s(22.6)。**专用推理服务**:TorchServe、Triton、BentoML、KServe、Seldon(自带批处理/版本/GPU 调度/A/B)。**同步 vs 异步**:IO 密集(调外部服务)用 async; CPU 密集推理用多进程 worker。面试金句:*"部署=把模型包成 REST API(FastAPI+pydantic 校验+uvicorn), 关键是模型启动时只加载一次、严格校验输入返回合适状态码、加健康检查探针、控延迟; 大规模用专用推理服务(Triton/BentoML/KServe)带批处理和版本管理, 再容器化上 K8s。"*
> **English**: **Model service = wrap the model in a REST API** (client POSTs feature JSON → returns prediction JSON), decoupling model from callers, language-agnostic, independently scalable. **FastAPI** advantages: pydantic auto-validates input, async high concurrency, auto Swagger docs, type safety. **Things to get right**: ① **load the model once at startup** (global/lifespan, never read from disk per request → else latency explodes) ② **strictly validate input** (missing/wrong-type/out-of-range → return 422, don't trust external input) ③ **health-check endpoint** (`/health` for K8s probes, readiness vs liveness) ④ **structured response + status codes** (200/422/500) ⑤ **low latency** (batch inference, async, model quantization). **Deployment stack**: FastAPI + **uvicorn/gunicorn** (ASGI server, multi-worker) + container (22.5) + K8s (22.6). **Dedicated inference servers**: TorchServe, Triton, BentoML, KServe, Seldon (built-in batching/versioning/GPU scheduling/A-B). **Sync vs async**: IO-bound (calling external services) use async; CPU-bound inference use multiple process workers. Interview line: *"Deployment = wrap the model in a REST API (FastAPI + pydantic validation + uvicorn); the keys are loading the model once at startup, strictly validating input with proper status codes, adding health probes, and controlling latency; at scale use dedicated inference servers (Triton/BentoML/KServe) with batching and versioning, then containerize onto K8s."*


In [ ]:

# ============================================================
# 从零实现一个 mini 模型服务(复现 FastAPI+pydantic 的核心)/ mini model service from scratch
# 中文:fastapi 本机没装, 我们用纯 Python 复现一个模型服务的核心逻辑:校验→预测→响应。真实 FastAPI 代码见下方。
# English: fastapi isn't installed here, so we reproduce a model service's core logic in pure Python: validate→predict→respond. Real FastAPI code below.
# ============================================================
import numpy as np, joblib, time
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris
X,y=load_iris(return_X_y=True)
Pipeline([("sc",StandardScaler()),("clf",LogisticRegression(max_iter=500))]).fit(X,y).__class__   # (train)
joblib.dump(Pipeline([("sc",StandardScaler()),("clf",LogisticRegression(max_iter=500))]).fit(X,y), "/tmp/iris_model.joblib")

FEATURES=["sepal_length","sepal_width","petal_length","petal_width"]
NAMES=list(load_iris().target_names)
class ValidationError(Exception): pass
def validate(payload):                                     # pydantic 干的活:校验 schema / what pydantic does: validate schema
    if not isinstance(payload, dict): raise ValidationError("payload 必须是对象 / must be an object")
    row=[]
    for f in FEATURES:
        if f not in payload: raise ValidationError(f"缺少字段 missing field: {f}")           # 缺字段 / missing
        v=payload[f]
        if not isinstance(v,(int,float)) or isinstance(v,bool):
            raise ValidationError(f"{f} 必须是数字 / must be a number")                       # 类型错 / wrong type
        if not (0<=v<=10): raise ValidationError(f"{f}={v} 超出合理范围[0,10] / out of range")  # 越界 / out of range
        row.append(float(v))
    return np.array(row).reshape(1,-1)

_MODEL=None
def startup():
    global _MODEL; _MODEL=joblib.load("/tmp/iris_model.joblib")   # ★ 启动时只加载一次 / load ONCE at startup
def health(): return {"status":200, "model_loaded":_MODEL is not None}   # /health 探活端点 / health probe
def predict(payload):                                                     # /predict 路由处理 / route handler
    t0=time.time()
    try: x=validate(payload)                                               # 先校验 / validate first
    except ValidationError as e: return {"status":422, "error":str(e)}     # 拒绝坏输入 / reject bad input (422)
    p=_MODEL.predict_proba(x)[0]; pred=int(_MODEL.predict(x)[0])
    return {"status":200, "prediction":pred, "class_name":str(NAMES[pred]),
            "confidence":round(float(p[pred]),3), "latency_ms":round((time.time()-t0)*1000,2)}

startup()                                                                 # 服务启动 / service starts
print("GET /health  →", health())
print("POST /predict 合法请求 / valid  →", predict({"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}))
print("POST /predict 缺字段 / missing   →", predict({"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4}))
print("POST /predict 类型错 / bad type  →", predict({"sepal_length":"oops","sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}))
print("POST /predict 越界   / range     →", predict({"sepal_length":999,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}))


**中文**:上面是原理复现。下面是**真实的 FastAPI 代码**——你在生产里就是这么写的(FastAPI 用 pydantic 的类型注解自动完成我们手写的那些校验):
**English**: The above reproduces the principles. Below is the **real FastAPI code** — this is exactly how you'd write it in production (FastAPI uses pydantic type hints to auto-do the validation we hand-wrote):

```python
# app.py  —— 真实生产代码 / real production code:  uvicorn app:app --host 0.0.0.0 --port 8000
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import joblib

app = FastAPI(title="Iris Classifier")
model = None

class IrisInput(BaseModel):                       # pydantic 自动校验:类型 + 范围 / auto validation: types + ranges
    sepal_length: float = Field(..., ge=0, le=10)
    sepal_width:  float = Field(..., ge=0, le=10)
    petal_length: float = Field(..., ge=0, le=10)
    petal_width:  float = Field(..., ge=0, le=10)

class Prediction(BaseModel):                      # 结构化响应 / structured response
    prediction: int
    class_name: str
    confidence: float

@app.on_event("startup")                          # ★ 启动时加载模型一次 / load model once at startup
def load_model():
    global model
    model = joblib.load("iris_model.joblib")

@app.get("/health")                               # K8s 探活 / health probe for K8s
def health():
    return {"status": "ok", "model_loaded": model is not None}

@app.post("/predict", response_model=Prediction)  # 校验由 pydantic 自动完成, 坏输入自动返回 422 / auto 422 on bad input
def predict(x: IrisInput):
    features = [[x.sepal_length, x.sepal_width, x.petal_length, x.petal_width]]
    proba = model.predict_proba(features)[0]
    pred = int(model.predict(features)[0])
    return Prediction(prediction=pred, class_name=NAMES[pred], confidence=float(proba[pred]))
```
**中文**:注意:我们手写的 `validate()` 那一大段,在 FastAPI 里只需一个 `IrisInput(BaseModel)` 类声明——**这就是框架的价值:把"必须做对的事"变成默认行为**。
**English**: Note: the whole `validate()` block we hand-wrote becomes just one `IrisInput(BaseModel)` class declaration in FastAPI — **that's the value of a framework: making "the things you must get right" the default behavior.**


In [ ]:

# ============================================================
# 可视化:请求生命周期 + 一个常见性能坑(每请求加载模型)/ request lifecycle + a common perf trap
# ============================================================
import matplotlib.pyplot as plt, time, joblib
# 演示:每请求从磁盘加载模型 vs 启动加载一次(用一个真实大小的模型:300 棵树的随机森林)/ per-request load vs load-once
from sklearn.ensemble import RandomForestClassifier
big=RandomForestClassifier(n_estimators=300, random_state=0).fit(X, y)   # 真实模型往往不小 / real models aren't tiny
joblib.dump(big, "/tmp/big_model.joblib")
_BIG=joblib.load("/tmp/big_model.joblib")                 # 启动时加载好的大模型 / preloaded at startup
xrow=validate({"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2})
def per_request_predict():
    m=joblib.load("/tmp/big_model.joblib")                # ✗ 每次请求都从磁盘反序列化整个模型 / reload from disk every request
    return m.predict(xrow)
def load_once_predict():
    return _BIG.predict(xrow)                             # ✓ 用启动时加载好的 / use the preloaded model
t=time.time()
for _ in range(100): per_request_predict()
t_bad=(time.time()-t)*1000/100
t=time.time()
for _ in range(100): load_once_predict()
t_good=(time.time()-t)*1000/100
fig,ax=plt.subplots(1,2,figsize=(14,5))
ax[0].bar(["✗ 每请求加载模型","✓ 启动时加载一次"],[t_bad,t_good],color=["#C44E52","#55A868"])
for i,v in enumerate([t_bad,t_good]): ax[0].text(i,v,f"{v:.2f}ms",ha="center",va="bottom",fontsize=11,weight="bold")
ax[0].set_ylabel("每请求延迟 ms"); ax[0].set_title(f"模型只加载一次 → 延迟降 {t_bad/t_good:.0f}x")
ax[1].axis("off"); ax[1].set_title("请求生命周期 / request lifecycle",fontsize=12,weight="bold")
steps=["① 接收 POST JSON","② pydantic 校验输入\n(坏输入→422)","③ 用预加载模型预测","④ 返回结构化 JSON+200"]
for i,s in enumerate(steps):
    ax[1].add_patch(plt.Rectangle((0.1,0.75-i*0.2),0.8,0.14,fc=["#DD8452","#4C72B0","#55A868","#9467BD"][i],alpha=0.3,transform=ax[1].transAxes))
    ax[1].text(0.5,0.82-i*0.2,s,ha="center",va="center",fontsize=9,transform=ax[1].transAxes)
    if i<3: ax[1].annotate("",xy=(0.5,0.73-i*0.2),xytext=(0.5,0.75-i*0.2),arrowprops=dict(arrowstyle="->"),transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/mlops03_viz.png",dpi=80); plt.show()
print(f"每请求加载模型 {t_bad:.2f}ms vs 启动加载一次 {t_good:.3f}ms → 快 {t_bad/t_good:.0f}x。这是最常见的部署性能坑")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **部署的核心不是"会写 FastAPI",而是"把几件必须做对的事做对"**:一个能跑的模型服务,90% 的功力在那几个不起眼的细节上:①**模型启动时只加载一次**——我们实测,即便是一个 300 棵树的中等模型,"每请求都从磁盘 `joblib.load`"也比"启动时加载一次"慢好几倍(把反序列化整个模型的开销加到了每一次预测上);而对真实的大模型(GB 级深度网络),每请求重新加载会给每次预测**平白加上几百毫秒甚至几秒**的延迟,是灾难性的。这是新手最常见的性能坑;②**永远校验外部输入**——缺字段、类型错、超范围的请求必须被明确拒绝(返回 422),而不是让模型崩溃、或悄悄给出对垃圾输入的垃圾预测;③**健康检查端点**——让 K8s/负载均衡器知道这个实例"活着且模型加载好了"。FastAPI 的价值,正是用 pydantic 类型注解把"校验输入"这件必须做对的事**变成一行声明的默认行为**。
2. **框架帮你做对,但你得知道它在帮你做什么**:我们手写了一大段 `validate()`,在 FastAPI 里只需一个 `BaseModel` 子类——但如果你不理解背后"为什么必须校验、为什么模型要启动时加载",你就算用了 FastAPI 也会写出每请求加载模型、不校验输入的烂服务。**框架降低了做对的门槛,但没消除理解的必要**。这也是为什么我们先从零实现再看框架:先懂原理,框架才是助力而非黑箱。
3. **诚实的边界:REST + FastAPI 是起点,不是终点**。①**它适合中小规模、单模型、CPU/轻量推理**;真到高吞吐、GPU、大模型、需要动态批处理(把多个请求合并成一个 batch 喂 GPU 以提升吞吐)时,你会需要**专用推理服务**——TorchServe、NVIDIA Triton、BentoML、KServe、Seldon,它们内置了批处理、多模型版本、GPU 调度、自动扩缩、A/B。②**FastAPI 服务本身还需要**:配 uvicorn/gunicorn 多 worker(单进程吃不满多核)、容器化(22.5)、上 K8s 做扩缩和自愈(22.6)、加监控(延迟/错误率/QPS)、限流、鉴权。③**同步 vs 异步**要分清:异步(`async`)只在 **IO 密集**(如调外部特征服务)时有意义,纯 CPU 的模型推理靠多进程 worker 才能并行。④**延迟优化**是大工程:模型量化(22.12)、缓存、批处理、更靠近用户的部署。**结论:FastAPI 让你 10 分钟把模型变成 API,但生产级模型服务是一个系统工程——理解请求生命周期、把'加载一次/严格校验/健康检查/低延迟'这些做对,再按规模选择专用推理框架,才是真正的部署能力。**

**English**:
1. **The core of deployment isn't "knowing FastAPI" but "getting a few must-get-right things right"**: 90% of a working model service's skill is in a few unassuming details: ① **load the model once at startup** — we measured that even for a modest 300-tree model, "`joblib.load` from disk per request" is several times slower than "load once at startup" (adding the cost of deserializing the whole model to every prediction); and for real large models (GB-scale deep nets), reloading per request adds hundreds of milliseconds or even seconds of latency to every prediction — catastrophic. This is the most common beginner performance trap; ② **always validate external input** — requests with missing fields, wrong types, out-of-range must be explicitly rejected (return 422), not crash the model or silently produce garbage predictions on garbage input; ③ **health-check endpoint** — so K8s/load balancer knows this instance is "alive with the model loaded." FastAPI's value is precisely making "validate input" — a must-get-right thing — **the default behavior of a one-line declaration** via pydantic type hints.
2. **The framework helps you get it right, but you must know what it's doing for you**: we hand-wrote a whole `validate()` block that becomes one `BaseModel` subclass in FastAPI — but if you don't understand "why validation is mandatory, why the model must load at startup," you'll write a bad service (per-request loading, no validation) even with FastAPI. **The framework lowers the bar to correctness but doesn't remove the need to understand**. That's why we implement from scratch first: understand the principles, and the framework becomes leverage rather than a black box.
3. **Honest limits: REST + FastAPI is a starting point, not the end**. ① **It suits small-to-medium scale, single-model, CPU/light inference**; at high throughput, GPU, large models, or needing dynamic batching (merging requests into one batch for the GPU to raise throughput), you'll need **dedicated inference servers** — TorchServe, NVIDIA Triton, BentoML, KServe, Seldon, with built-in batching, multi-model versioning, GPU scheduling, autoscaling, A/B. ② **The FastAPI service itself also needs**: uvicorn/gunicorn multi-worker (one process can't saturate many cores), containerization (22.5), K8s for scaling and self-healing (22.6), monitoring (latency/error rate/QPS), rate limiting, authentication. ③ **Sync vs async** must be distinguished: async (`async`) matters only for **IO-bound** work (e.g. calling an external feature service); pure-CPU model inference parallelizes via multiple process workers. ④ **Latency optimization** is a big effort: model quantization (22.12), caching, batching, deploying closer to users. **Conclusion: FastAPI turns a model into an API in 10 minutes, but a production-grade model service is systems engineering — understand the request lifecycle, get "load once / strict validation / health checks / low latency" right, then choose a dedicated inference framework by scale; that's real deployment capability.**

> 💼 **实战视角 / Practical angle**
> **中文**:FastAPI 部署落地:①**结构**:`@app.on_event("startup")` 加载模型一次 → pydantic `BaseModel` 声明输入(自动校验+文档)→ `/predict` 处理 → `/health` 探活;②**跑起来**:`uvicorn app:app` 开发, 生产用 `gunicorn -k uvicorn.workers.UvicornWorker -w 4`(多 worker 吃满多核);③**容器化**(22.5)+ K8s(22.6)做扩缩自愈;④**监控**延迟 p50/p99、错误率、QPS;⑤**大规模/GPU**换专用推理服务(Triton 动态批处理、BentoML 打包、KServe on K8s);⑥安全:鉴权、限流、输入大小限制。**别犯的错**:每请求加载模型、不校验输入、把重预处理写在请求路径里(应在 Pipeline 内, 接 22.2)、同步阻塞调用。面试金句:*"FastAPI+pydantic 把模型包成 REST API:启动时加载模型一次、pydantic 自动校验输入(坏输入返回422)、加 /health 探针、结构化响应; 用 uvicorn/gunicorn 多 worker + 容器 + K8s 扩缩; 高吞吐/GPU 换 Triton/BentoML/KServe 拿动态批处理和版本管理; 关键是低延迟和不信任外部输入。"*
> **English**: FastAPI deployment in practice: ① **structure**: `@app.on_event("startup")` loads the model once → pydantic `BaseModel` declares input (auto validation + docs) → `/predict` handler → `/health` probe; ② **run it**: `uvicorn app:app` for dev, production uses `gunicorn -k uvicorn.workers.UvicornWorker -w 4` (multi-worker to saturate cores); ③ **containerize** (22.5) + K8s (22.6) for scaling and self-healing; ④ **monitor** latency p50/p99, error rate, QPS; ⑤ **at scale/GPU** switch to dedicated inference servers (Triton dynamic batching, BentoML packaging, KServe on K8s); ⑥ security: auth, rate limiting, input size limits. **Don't**: load the model per request, skip input validation, put heavy preprocessing in the request path (should be in the Pipeline, per 22.2), make synchronous blocking calls. Interview line: *"FastAPI + pydantic wraps a model in a REST API: load the model once at startup, pydantic auto-validates input (422 on bad input), add a /health probe, structured responses; run uvicorn/gunicorn multi-worker + container + K8s for scaling; at high throughput/GPU switch to Triton/BentoML/KServe for dynamic batching and versioning; the keys are low latency and not trusting external input."*

---
### 小结 / Summary
- **中文**:部署=把模型包成 REST API(FastAPI+pydantic); 请求生命周期:接收→校验→预测→响应。
- **English**: Deployment = wrap the model in a REST API (FastAPI + pydantic); request lifecycle: receive → validate → predict → respond.
- **中文**:必做对:模型启动时加载一次(否则延迟爆炸)、严格校验输入(坏输入返回422)、健康检查端点、低延迟。
- **English**: Must get right: load model once at startup (else latency explodes), strictly validate input (422 on bad input), health-check endpoint, low latency.
- **中文**:uvicorn/gunicorn 多 worker + 容器 + K8s; 高吞吐/GPU 用专用推理服务(Triton/BentoML/KServe)带动态批处理和版本。
- **English**: uvicorn/gunicorn multi-worker + container + K8s; at high throughput/GPU use dedicated inference servers (Triton/BentoML/KServe) with dynamic batching and versioning.
